# Reproduce — LOH arm headline (B\*58:01 allelic loss bias)

**Claim reproduced.** Across the MSK-IMPACT pan-cancer HLA-LOH cohort, **B\*58:01** is
preferentially *lost* (rather than retained) in within-patient heterozygous LOH events:
`frac_lost = 0.665`, two-sided **exact binomial** p vs 0.5 = `3.0e-5`, **BH-FDR = 0.003**
(global correction across 110 alleles; the marginal-binomial multiplicity estimand of the arm).

This notebook is **self-contained**: it reads only the committed, aggregated, non-identifiable
per-allele table `../data/derived/loh_per_allele_bias.csv` (110 rows, allele-level counts only —
no patient rows) and recomputes the statistic from the raw `n_lost` / `n_total` counts. No source
event table, no patient data.


In [1]:
# Pinned environment (versions this run artifact was executed under):
#   python==3.11.15
#   numpy==2.4.6
#   pandas==2.3.3
#   scipy==1.17.1
#   statsmodels==0.14.6
import sys, numpy, pandas, scipy, statsmodels
print("python     ", sys.version.split()[0])
print("numpy      ", numpy.__version__)
print("pandas     ", pandas.__version__)
print("scipy      ", scipy.__version__)
print("statsmodels", statsmodels.__version__)

python      3.11.15
numpy       2.4.6
pandas      2.3.3
scipy       1.17.1
statsmodels 0.14.6


In [2]:
import pandas as pd
from scipy.stats import binomtest
from statsmodels.stats.multitest import multipletests

# Read ONLY the committed derived table, by relative path.
df = pd.read_csv("../data/derived/loh_per_allele_bias.csv")
print("rows:", len(df), "| columns:", list(df.columns))
df.head()

rows: 110 | columns: ['allele', 'locus', 'n_lost', 'n_retained', 'n_total', 'frac_lost', 'p', 'FDR']


    allele locus  n_lost  n_retained  n_total  frac_lost         p       FDR
0  B*58:01     B     109          55      164   0.664634  0.000030  0.003294
1  B*08:01     B     320         424      744   0.430108  0.000156  0.008563
2  B*07:02     B     371         474      845   0.439053  0.000443  0.016256
3  B*51:01     B     251         186      437   0.574371  0.002166  0.059567
4  A*01:01     A     636         748     1384   0.459538  0.002835  0.062362

In [3]:
# Recompute the two-sided exact binomial vs p=0.5 for every allele, from raw counts.
p_recomputed = [
    binomtest(int(r.n_lost), int(r.n_total), p=0.5, alternative="two-sided").pvalue
    for r in df.itertuples()
]
# Global BH-FDR across all 110 alleles (the arm's marginal-binomial multiplicity estimand).
fdr_recomputed = multipletests(p_recomputed, method="fdr_bh")[1]

df_chk = df.assign(
    frac_lost_recomputed = df.n_lost / df.n_total,
    p_recomputed = p_recomputed,
    FDR_recomputed = fdr_recomputed,
)
row = df_chk.loc[df_chk.allele == "B*58:01"].iloc[0]
print(f"B*58:01: n_lost={int(row.n_lost)}  n_total={int(row.n_total)}")
print(f"  frac_lost   recomputed = {row.frac_lost_recomputed:.4f}   (reported {row.frac_lost:.4f})")
print(f"  binomial p  recomputed = {row.p_recomputed:.3e}   (reported {row.p:.3e})")
print(f"  BH-FDR      recomputed = {row.FDR_recomputed:.4f}   (reported {row.FDR:.4f})")

B*58:01: n_lost=109  n_total=164
  frac_lost   recomputed = 0.6646   (reported 0.6646)
  binomial p  recomputed = 2.995e-05   (reported 2.995e-05)
  BH-FDR      recomputed = 0.0033   (reported 0.0033)


In [4]:
# Assert the recomputed headline matches the reported values.
import numpy as np

assert abs(row.frac_lost_recomputed - 0.665) < 5e-4, row.frac_lost_recomputed
assert abs(row.frac_lost_recomputed - row.frac_lost) < 1e-9
assert abs(row.p_recomputed - 2.995e-5) < 1e-7, row.p_recomputed
assert abs(row.FDR_recomputed - 0.003) < 5e-4, row.FDR_recomputed
assert abs(row.FDR_recomputed - row.FDR) < 1e-9
# whole-table consistency: recomputed p and FDR match the committed columns everywhere
assert np.allclose(df_chk.p, df_chk.p_recomputed, atol=1e-12)
assert np.allclose(df_chk.FDR, df_chk.FDR_recomputed, atol=1e-12)
print("PASS — B*58:01 frac_lost 0.665, exact-binomial p 3.0e-5, BH-FDR 0.003 reproduced from committed CSV.")

PASS — B*58:01 frac_lost 0.665, exact-binomial p 3.0e-5, BH-FDR 0.003 reproduced from committed CSV.


**Result.** The headline reproduces exactly from the committed aggregated table: B\*58:01 shows
`frac_lost = 0.6646`, two-sided exact-binomial `p = 2.995e-5`, and global BH-FDR = `0.0033` across
110 alleles. B\*58:01 is the **most statistically significant** loss-biased class I allele in the cohort —
the lowest FDR and one of only three alleles surviving FDR < 0.05 on the marginal-binomial estimand
(the other two — B\*08:01 and B\*07:02 — are *retention*-biased). By raw magnitude it is not the
highest `frac_lost` (several small-n alleles rank higher, e.g. A\*36:01 0.710 at n=31); its
distinction is the significance of its loss bias at high event count (n=164), not the raw fraction.